# Une ligne de plus, une autre histoire · *One more line, another story*

Notebook compagnon du chapitre **19. Premier script Python : charger une série FRED et la tracer en dix lignes** — [lire l'article](https://nmlab.io/ressources/premier-script-python-fred).
Companion notebook to chapter **19. Your First Python Script: Load a FRED Series and Plot It in Ten Lines** — [read the article](https://nmlab.io/en/ressources/first-python-script-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_gdp_growth() -> Series:
    """Croissance du PIB réel sur un an / real GDP year-over-year growth.
    pct_change(4) car le PIB est trimestriel / pct_change(4) since GDP is quarterly."""
    gdp = nm.load_fred("GDPC1")
    return (gdp.pct_change(4) * 100).dropna()

growth = load_gdp_growth()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Une ligne de plus, une autre histoire",
        sub="La même série, transformée en glissement annuel par « pct_change »",
        ylab="croissance du PIB réel sur un an, %",
        covid="COVID : −7,5 %\npuis +12,4 %",
        note="df.pct_change(4)*100 : une instruction, et le niveau devient un taux de croissance — krach COVID et\n"
             "rebond compris. C'est là que le code dépasse la souris. Source : BEA via FRED (GDPC1)."),
    "en": dict(
        title="One more line, another story",
        sub="The same series, turned into year-over-year growth by « pct_change »",
        ylab="real GDP growth year over year, %",
        covid="COVID: −7.5%\nthen +12.4%",
        note="df.pct_change(4)*100: one instruction, and the level becomes a growth rate — COVID crash and\n"
             "rebound included. That is where code beats the mouse. Source: BEA via FRED (GDPC1)."),
}

def build_figure(growth: Series, lang: str) -> Figure:
    """Glissement annuel : ligne claire, aire bleue (>0) et rose (<0)."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1026)
    ax = nm.axes(fig)
    ax.fill_between(growth.index, growth, 0, where=growth >= 0,
                    color=nm.COLORS["blue"], alpha=0.5, interpolate=True)
    ax.fill_between(growth.index, growth, 0, where=growth < 0,
                    color=nm.COLORS["rose"], alpha=0.5, interpolate=True)
    ax.plot(growth.index, growth, color=nm.COLORS["text"], linewidth=1.6)
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.4, alpha=0.9)
    ax.set_ylim(-10, 14)
    ax.set_yticks(range(-10, 11, 5))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    # Annotation sur le creux COVID de 2020 / annotate the 2020 COVID trough
    trough = growth.loc["2020"].idxmin()
    ax.annotate(text["covid"], xy=(trough, growth.loc[trough]), xytext=(-150, 34),
                textcoords="offset points", ha="right", va="center", fontsize=22,
                fontweight="bold", color=nm.COLORS["rose"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["rose"], lw=1.8))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(growth, LANG)